# Apache Spark Memory Management: Complete Mastery Guide
## From First Principles to Production Configurations

## Module 1: Core Memory Architecture
### 1.1 The Big Picture
```mermaid
graph TD
    A[Total Container Memory] --> B[Executor Memory]
    A --> C[Overhead Memory]
    B --> D[JVM Heap]
    D --> E[Reserved Memory]
    D --> F[User Memory]
    D --> G[Unified Memory]
    G --> H[Execution Memory]
    G --> I[Storage Memory]
    C --> J[Off-Heap Memory]
    C --> K[Python Worker Memory]
```

### 1.2 Memory Allocation Formula
For `--executor-memory 2G`:

```
Total Container Memory = Executor Memory + Overhead Memory
                      = 2GB + MAX(384MB, 10% of 2GB)
                      = 2048MB + 384MB = 2432MB

JVM Heap Breakdown:
Reserved Memory = 300MB (fixed)
Usable Memory = 2048MB - 300MB = 1748MB

Unified Memory = 1748MB * 0.6 = 1048.8MB
User Memory = 1748MB * 0.4 = 699.2MB

Execution vs Storage Split:
Initial Allocation = 1048.8MB / 2 = 524.4MB each
```

**Visual Representation:**
```
+----------------------------+
|  Container (2432MB)        |
|  +----------------------+  |
|  | JVM Heap (2048MB)    |  |
|  | +------------------+ |  |
|  | | Reserved 300MB   | |  |
|  | +------------------+ |  |
|  | | Unified 1049MB   | |  |
|  | |  ↳ Exec 524MB    | |  |
|  | |  ↳ Storage 524MB | |  |
|  | +------------------+ |  |
|  | | User 699MB       | |  |
|  | +------------------+ |  |
|  +----------------------+  |
|  | Overhead 384MB      |  |
|  +----------------------+  |
+----------------------------+
```

## Module 2: Dynamic Memory Behavior
### 2.1 Memory Eviction Scenarios

**Case 1: Storage Demands More Space**
```
Initial State:
Execution: 524MB (100MB used)
Storage: 524MB (400MB used)

New Cache Request: 600MB
Available Storage: 524MB - 400MB = 124MB
Execution Free: 524MB - 100MB = 424MB

Result:
Storage borrows 424MB from Execution
New Storage: 524MB + 424MB = 948MB
Execution: 524MB - 424MB = 100MB
```

**Case 2: Execution Demands More Space**
```
Initial State:
Execution: 524MB (500MB used)
Storage: 524MB (200MB used)

New Shuffle Operation: Needs 700MB
Execution Free: 524MB - 500MB = 24MB
Storage Free: 524MB - 200MB = 324MB

Result:
Execution evicts 324MB from Storage
New Execution: 524MB + 324MB = 848MB
Storage: 524MB - 324MB = 200MB
```

## Module 3: Configuration Deep Dive
### 3.1 Key Configuration Parameters

| Parameter | Default | Effect | Production Tuning |
|-----------|---------|--------|--------------------|
| `spark.executor.memory` | 1g | Base JVM heap | Set to 60-75% of container |
| `spark.memory.fraction` | 0.6 | Unified memory ratio | Increase to 0.8 for analytics |
| `spark.memory.storageFraction` | 0.5 | Storage protection | Lower to 0.3 for ETL jobs |
| `spark.executor.memoryOverhead` | MAX(384M, 0.1*executor) | Off-JVM memory | Add 1-2GB for PySpark |

**Sample Tuned Configuration:**
```bash
spark-submit \
--executor-memory 8G \
--conf spark.memory.fraction=0.8 \
--conf spark.memory.storageFraction=0.3 \
--conf spark.executor.memoryOverhead=2G \
--conf spark.executor.pyspark.memory=1G \
app.py
```

**Resulting Allocation:**
```
JVM Heap = 8GB
Reserved = 300MB
Unified = (8192MB - 300MB) * 0.8 = 6314MB
User = (8192MB - 300MB) * 0.2 = 1578MB
Storage Protected = 6314MB * 0.3 = 1894MB
Execution Minimum = 6314MB - 1894MB = 4420MB
```

## Module 4: Special Memory Areas
### 4.1 Off-Heap Memory Mechanics

**Enabled with:**
```python
SparkSession.builder \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g")
```

**Memory Hierarchy:**
```mermaid
graph LR
    A[Operating System] --> B[JVM Heap]
    A --> C[Off-Heap Memory]
    C --> D[Spark Managed]
    C --> E[Direct Buffers]
```

**Advantages:**
- Avoid Garbage Collection pauses
- Share memory between JVM and native code
- Store serialized data more efficiently

### 4.2 PySpark Memory Considerations

**Python Worker Architecture:**
```
+-------------------+
| Executor JVM      |
|  - Java Objects   |
|  - Block Manager  |
+-------------------+
        ⇅ IPC
+-------------------+
| Python Worker     |
|  - NumPy Arrays   |
|  - Pandas DF      |
+-------------------+
```

**Critical Configurations:**
```bash
# For ML workloads
--conf spark.executor.memoryOverhead=4G \
--conf spark.executor.pyspark.memory=2G \
--conf spark.sql.execution.arrow.pyspark.enabled=true
```

**Memory Flow:**
1. JVM stores data in serialized form
2. Python process deserializes into native objects
3. Results serialized back to JVM for shuffling

## Module 5: Troubleshooting Guide
### 5.1 Common Errors & Solutions

**Error 1: Container Memory Exceeds YARN Limits**
```
Required executor memory (8192+1024MB)
exceeds yarn.scheduler.maximum-allocation-mb (8192MB)
```

**Solution:**
```bash
# Reduce overhead memory
--conf spark.executor.memoryOverhead=512M

# Or increase YARN configs
<property>
  <name>yarn.scheduler.maximum-allocation-mb</name>
  <value>16384</value>
</property>
```

**Error 2: Frequent GC Pauses**
```
java.lang.OutOfMemoryError: GC overhead limit exceeded
```

**Solution:**
```bash
# Enable off-heap
--conf spark.memory.offHeap.enabled=true
--conf spark.memory.offHeap.size=2g

# Tune JVM
--conf spark.executor.extraJavaOptions="-XX:+UseG1GC"
```

### 5.2 Monitoring Techniques

**Spark UI Metrics:**
```mermaid
graph TD
    A[Spark UI] --> B[Storage Tab]
    A --> C[Executors Tab]
    B --> D[Cache Usage]
    C --> E[Heap/Off-Heap Usage]
    C --> F[GC Time]
```

**Critical JMX Metrics:**
```
HeapMemoryUsage.used
NonHeapMemoryUsage.used
GarbageCollector.*.count
```

**Sample Profiling Command:**
```bash
jstat -gc <executor-pid> 1000
```